# Valutazione finale: tutti gli esperimenti rispetto al top-down

La tabella riepilogativa contiene Precision, Recall e F1 medie di HLG e LLG per il top-down originale e ogni configurazione della campagna selezionata. Per default viene scelta la suite piu recente; SUITE_ID permette di selezionarne un'altra senza mescolare versioni.

Si valuta la gerarchia finale dopo tutte le iterazioni di ciascuna run completata, anche senza convergenza. Non si mediano gli stati intermedi. Per ogni dataset si mediano prima le ripetizioni finali; poi si calcola separatamente la media aritmetica di Precision, Recall e F1 tra i dataset, con lo stesso peso per ciascuno. La F1 media e la media delle F1, non la F1 ricavata da Precision e Recall medie.

I metodi con risultati disponibili vengono confrontati sugli stessi dataset e sulle stesse baseline. I metodi ancora senza risultati rimangono in tabella con metriche mancanti e non restringono la coorte. Le colonne di copertura segnalano campagne parziali: la tabella definitiva richiede il completamento delle run previste. Nessuna generazione viene avviata e nessun risultato mancante viene sostituito con zero.

Una seconda tabella unica riporta ogni esperimento per dataset, con Precision, Recall e F1 di HLG e LLG e i sei delta rispetto alla baseline top-down corrispondente. Usa tutti i risultati finali disponibili, anche fuori dalla coorte comune del riepilogo; le ripetizioni sono mediate entro dataset.

Una terza tabella mostra convergenza, iterazioni, motivo di arresto e conteggi HLG/LLG top-down e bottom-up per ciascuna run, mantenendo separate le ripetizioni.

Una quarta tabella conta le decisioni valide di ciascun LLM per esperimento, dataset e ripetizione, con esempi, temperatura e diagnostica dei tentativi scartati.

La configurazione **reference** (4 esempi; temperatura letta dal manifest: 0,2 nel nuovo protocollo) viene inclusa insieme alle altre quattro configurazioni della suite. Gli output del notebook 02 non vengono caricati: il metodo separato ?Bottom-up originale? non compare nel confronto. I file salvati restano intatti.

La valutazione seleziona la suite piu recente (`SUITE_ID = None`); la lettura di formati storici e disabilitata per questa esecuzione.

Il confronto generale usa esattamente tutti i dodici dataset e soltanto configurazioni interamente concluse. La vista delle configurazioni parziali ? conservata separatamente e non determina la coorte della tabella generale.


In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
for directory in (PROJECT_ROOT, PROJECT_ROOT / "notebook"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

from groundtruth import GROUNDTRUTH_BY_NAME

# None seleziona la suite piu recente; impostare un ID per una campagna specifica.
SUITE_ID = None
# Explicit read-only access to this historical schema-2 campaign only.
HISTORICAL_SUITE_ID = None
EXPERIMENTS_ROOT = PROJECT_ROOT / "output" / "bottom_up_experiments"
available_suites = sorted(p.parent for p in EXPERIMENTS_ROOT.glob("*/manifest.json"))
if not available_suites:
    raise ValueError("Nessuna suite sperimentale salvata")
SUITE_DIRS = [EXPERIMENTS_ROOT / SUITE_ID] if SUITE_ID else [available_suites[-1]]
REPORT_ID = datetime.now(ZoneInfo("Europe/Rome")).strftime("%Y%m%d_%H%M%S_%f")
REPORT_DIR = PROJECT_ROOT / "output" / "evaluation" / "configuration_comparison" / REPORT_ID
print("Suite:", SUITE_DIRS[0].name)
print("Report:", REPORT_DIR)


Suite: ofat_fixed_thresholds_20261025_170859
Report: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\configuration_comparison\20261026_230638_419662


## Lettura degli output salvati


In [2]:
from collections import Counter
import hashlib
import json
from pathlib import Path


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def baseline_key(payload):
    fields = ("description", "structuredHighLevelGoals", "structuredLowLevelGoals")
    if any(field not in payload for field in fields):
        raise ValueError("Missing verifiable initial hierarchy or description")
    return digest(json.dumps({field: payload[field] for field in fields},
                             sort_keys=True, ensure_ascii=False).encode("utf-8"))


def goal_texts(container, field):
    from src.evaluation.goal_evaluator import goal_descriptions
    return goal_descriptions(container, field)




In [3]:
def historical_result_payload(document, record):
    """Read schema-2 traces for tables in memory; never rewrite historical JSON."""
    from copy import deepcopy
    cycle = deepcopy(record["globalGoalCycle"])
    allowed = {"KEEP_ORIGINAL_HIGH_LEVEL_GOAL", "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL",
               "REMOVE_ORIGINAL_HIGH_LEVEL_GOAL", "DUPLICATE_REMOVAL"}
    for trace in cycle.get("iterations", []):
        evaluations = {}
        for branch, old in trace.get("evaluations", {}).items():
            high = old.get("high_level_evaluation") or {}
            action = high.get("decision")
            if old.get("final_decision") == "CONFIRM_BRANCH":
                action = "KEEP_ORIGINAL_HIGH_LEVEL_GOAL"
            if old.get("evaluation_status") == "INCONCLUSIVE" or action not in allowed:
                action = None
            evaluations[branch] = dict(branch_id=branch, decision=action,
                evaluation_status="DECIDED" if action else "INCONCLUSIVE",
                feedback=old.get("feedback") if action == "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL" else None,
                surviving_branch_id=high.get("covered_by_branch_id") if action == "DUPLICATE_REMOVAL" else None)
        trace["evaluations"] = evaluations
        trace["evaluator_responses"] = [dict(model=vote["model"], status=vote["status"],
            decisions=(vote.get("parsed_response") or {}).get("decisions", []) if vote["status"] == "VALID" else [],
            attempts=[{"generation": attempt} for attempt in vote.get("generation_attempts", [])])
            for vote in trace.get("critic1_global_votes", [])]
    return {**deepcopy(document["baseline"]), "globalGoalCycle": cycle,
            "finalHighLevelGoals": record["finalHighLevelGoals"],
            "finalLowLevelGoals": record["finalLowLevelGoals"], "source_schema_version": 2}


def collect_outputs(project_root, suite_directories, *, historical_suite_id=None):
    """Keep planned/missing runs visible; read actual JSON results, never old CSVs."""
    # Notebook kernels retain imported modules after source files are updated.
    # Refresh the shared schema before storage so both use the current contract.
    import importlib
    import src.data_model as data_model
    import src.bottom_up.result_storage as result_storage
    importlib.invalidate_caches()
    importlib.reload(data_model)
    importlib.reload(result_storage)
    from src.bottom_up.result_storage import payload_hash, result_payload, require_current_schema
    root = Path(project_root)
    hashes, entries, availability, baselines = {}, [], [], {}
    configurations = []

    def read(path):
        path = Path(path).resolve()
        raw = path.read_bytes()
        checksum = digest(raw)
        if path in hashes and hashes[path] != checksum:
            raise ValueError(f"Input changed during collection: {path}")
        hashes[path] = checksum
        return json.loads(raw)

    def register_baseline(payload, path):
        key = baseline_key(payload)
        identity = (payload["name"], key)
        if identity not in baselines:
            entry = {"dataset": payload["name"], "baseline_key": key, "suite_id": "",
                     "method": "top_down", "config_id": "top_down", "repetition": 0,
                     "source_file": str(path), "payload": payload,
                     "hlg": payload["structuredHighLevelGoals"], "llg": payload["structuredLowLevelGoals"]}
            goal_texts(entry["hlg"], "goals")
            goal_texts(entry["llg"], "low_level_goals")
            entries.append(entry)
            baselines[identity] = entry
            availability.append({k: v for k, v in entry.items() if k not in {"payload", "hlg", "llg"}} | {"status": "completed"})
        return key

    for path in sorted((root / "output" / "top_down").glob("*_top_down.json")):
        register_baseline(read(path), path)
    for directory in map(Path, suite_directories):
        manifest = read(directory / "manifest.json")
        historical = (directory.name == historical_suite_id
                      and manifest.get("schema_version") == 2
                      and manifest.get("storage_layout") == "dataset_json_v2")
        if not historical:
            require_current_schema(manifest)
        if not historical and manifest.get("storage_layout") != "dataset_json_v3":
            raise ValueError("Unsupported suite layout; prepare a new schema-3 suite")
        selected_configs = manifest["configurations"]
        configurations.extend({"suite_id": manifest["suite_id"],
                               "design": manifest.get("design", "unspecified"),
                               "planned_datasets": len(manifest["baselines"]),
                               "planned_repetitions": manifest["repetitions"], **config}
                              for config in selected_configs)
        for baseline in manifest["baselines"]:
            payload = baseline["payload"]
            if payload_hash(payload) != baseline["sha256"]:
                raise ValueError("Frozen compact baseline changed")
            key = register_baseline(payload, directory / "manifest.json")
            path = directory / baseline["dataset_file"]
            document = read(path) if path.exists() else None
            if document is not None:
                if historical:
                    if document.get("schema_version") != 2:
                        raise ValueError("Historical dataset schema differs from manifest")
                else:
                    require_current_schema(document)
            if document and (document["baseline_sha256"] != baseline["sha256"]
                             or payload_hash(document["baseline"]) != baseline["sha256"]):
                raise ValueError("Compact dataset baseline differs from the manifest")
            for config in selected_configs:
                for repetition in range(1, manifest["repetitions"] + 1):
                    run_id = f"{config['config_id']}/repeat_{repetition:03d}"
                    record = document["runs"].get(run_id, {}) if document else {}
                    identity = dict(dataset=baseline["dataset"], baseline_key=key,
                        suite_id=manifest["suite_id"], config_id=config["config_id"],
                        method=f"{manifest['suite_id']}/{config['config_id']}", repetition=repetition)
                    row = {**identity, "status": record.get("status", "not_started"),
                           "source_file": str(path), "error": record.get("error")}
                    if row["status"] == "completed":
                        if record["experiment"] != config or record["repetition"] != repetition:
                            raise ValueError("Compact run configuration differs from the manifest")
                        result = historical_result_payload(document, record) if historical else result_payload(document, record)
                        goal_texts(result["finalHighLevelGoals"], "goals")
                        goal_texts(result["finalLowLevelGoals"], "low_level_goals")
                        entries.append({**identity, "source_file": str(path), "payload": result,
                                        "hlg": result["finalHighLevelGoals"], "llg": result["finalLowLevelGoals"]})
                    availability.append(row)
    return {"entries": entries, "availability": availability, "input_hashes": hashes,
            "configurations": configurations}




In [4]:
METRICS = ["precision", "recall", "f1"]


def evaluate_final_outputs(collection, references, score_goals):
    """Score each final hierarchy once; intermediate iterations are never averaged."""
    rows, matches = [], []
    for entry in collection["entries"]:
        identity = {k: entry[k] for k in (
            "dataset", "baseline_key", "suite_id", "method", "config_id", "repetition", "source_file")}
        if entry["dataset"] not in references:
            raise ValueError(f"Missing ground truth: {entry['dataset']}")
        cycle = entry["payload"].get("globalGoalCycle", {})
        for level, field, ref_field in (("HLG", "goals", "highLevelGoals"),
                                       ("LLG", "low_level_goals", "lowLevelGoals")):
            predictions = goal_texts(entry[level.lower()], field)
            reference = references[entry["dataset"]][ref_field]
            result = dict(score_goals(predictions, reference))
            assigned = result.pop("matches", [])
            rows.append({**identity, "level": level,
                "precision": result["precision"], "recall": result["recall"],
                "f1": result.get("f1", result.get("f1_score")),
                "generated_goal_count": len(predictions), "reference_goal_count": len(reference),
                "completed_iterations": cycle.get("completed_iterations"),
                "converged": cycle.get("converged"), "stop_reason": cycle.get("stop_reason")})
            matches.extend({**identity, "level": level, **match} for match in assigned)
    columns = ["dataset", "baseline_key", "suite_id", "method", "config_id", "repetition",
               "source_file", "level", *METRICS, "generated_goal_count", "reference_goal_count",
               "completed_iterations", "converged", "stop_reason"]
    return pd.DataFrame(rows, columns=columns), pd.DataFrame(matches)


def final_comparison_table(metrics, configurations, availability):
    """Macro means over final results on a common cohort, repetitions averaged first."""
    methods = ["top_down"] + list(dict.fromkeys(
        f"{c['suite_id']}/{c['config_id']}" for c in configurations))
    datasets = metrics.groupby(["method", "dataset", "baseline_key", "level"], dropna=False)[METRICS].mean().reset_index()
    # A dataset is eligible only when both HLG and LLG were evaluated.
    cohorts = {}
    for method in methods:
        group = datasets[datasets.method == method]
        counts = group.groupby(["dataset", "baseline_key"])["level"].nunique()
        cohorts[method] = set(counts[counts == 2].index)
    active = [method for method in methods if method != "top_down" and cohorts[method]]
    common = set(cohorts["top_down"])
    for method in active:
        common &= cohorts[method]
    if common:
        names = [name for name, _ in common]
        if len(names) != len(set(names)):
            raise ValueError("Multiple baseline versions for one dataset; select one compatible suite")
    mask = pd.Series([tuple(row) in common for row in datasets[["dataset", "baseline_key"]].to_numpy()], index=datasets.index, dtype=bool)
    selected = datasets.loc[mask]
    configuration_map = {f"{c['suite_id']}/{c['config_id']}": c for c in configurations}
    rows = []
    for method in methods:
        frame = selected[selected.method == method]
        raw = metrics[(metrics.method == method) & metrics.level.eq("HLG")]
        runs_used = sum((d, b) in common for d, b in raw[["dataset", "baseline_key"]].itertuples(index=False, name=None))
        row = {"Esperimento": "Top-down originale" if method == "top_down" else (
            "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[1]),
            "Dataset valutati": len(common & cohorts[method]),
            "Dataset disponibili": len(cohorts[method]), "Run valutate": runs_used}
        for level in ("HLG", "LLG"):
            averages = frame[frame.level == level][METRICS].mean()
            for metric, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                row[f"{level} {label}"] = averages[metric]
        config = configuration_map.get(method, {})
        row["Esempi evaluator"] = config.get("evaluator_few_shot_examples")
        row["Temperatura evaluator"] = config.get("evaluator_temperature")
        planned = [r for r in availability if r.get("method") == method]
        row["Stato"] = ("Nessun risultato finale" if not cohorts[method] else
            "Nessun dataset comune" if not common else
            "Parziale" if any(r.get("status") != "completed" for r in planned) else "Disponibile")
        rows.append(row)
    columns = ["Esperimento", "Esempi evaluator", "Temperatura evaluator", "HLG Precision", "HLG Recall", "HLG F1", "LLG Precision", "LLG Recall", "LLG F1",
               "Dataset valutati", "Dataset disponibili", "Run valutate", "Stato"]
    table = pd.DataFrame(rows, columns=columns)
    for level in ("HLG", "LLG"):
        for metric in ("Precision", "Recall", "F1"):
            column = f"{level} {metric}"
            table[f"{level} Delta {metric}"] = table[column] - table.loc[0, column]
    metric_columns = [column for level in ("HLG", "LLG")
                      for column in ([f"{level} {m}" for m in ("Precision", "Recall", "F1")]
                                     + [f"{level} Delta {m}" for m in ("Precision", "Recall", "F1")])]
    return table[["Esperimento", "Esempi evaluator", "Temperatura evaluator", *metric_columns,
                  "Dataset valutati", "Dataset disponibili", "Run valutate", "Stato"]], common


def dataset_comparison_table(metrics, configurations, availability):
    """One row per method/dataset/baseline, paired with its own top-down result."""
    methods = ["top_down"] + list(dict.fromkeys(
        f"{c['suite_id']}/{c['config_id']}" for c in configurations))
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in configurations}
    identity = ["method", "dataset", "baseline_key", "level"]
    means = metrics.groupby(identity, dropna=False)[METRICS].mean()
    counts = metrics.groupby(identity, dropna=False)["repetition"].nunique()
    keys = set(metrics[["dataset", "baseline_key"]].itertuples(index=False, name=None))
    keys.update((r["dataset"], r["baseline_key"]) for r in availability
                if r.get("baseline_key") is not None)
    rows = []
    for method in methods:
        config = configs.get(method, {})
        for dataset, baseline in sorted(keys):
            planned = [r for r in availability if r.get("method") == method
                       and r.get("dataset") == dataset and r.get("baseline_key") == baseline]
            row = {"Dataset": dataset, "Esperimento": "Top-down originale" if method == "top_down" else (
                "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[1]),
                "Suite": config.get("suite_id", ""), "Baseline": baseline,
                "Esempi evaluator": config.get("evaluator_few_shot_examples"),
                "Temperatura evaluator": config.get("evaluator_temperature")}
            level_counts, paired = [], True
            for level in ("HLG", "LLG"):
                key = (method, dataset, baseline, level)
                reference = ("top_down", dataset, baseline, level)
                level_counts.append(int(counts.get(key, 0)))
                paired = paired and reference in means.index
                for metric, label in (("precision", "Precision"), ("recall", "Recall"), ("f1", "F1")):
                    value = means.loc[key, metric] if key in means.index else float("nan")
                    base = means.loc[reference, metric] if reference in means.index else float("nan")
                    row[f"{level} {label}"] = value
                    row[f"{level} Delta {label}"] = value - base
            row["Ripetizioni valutate"] = min(level_counts)
            expected = config.get("planned_repetitions", 1)
            row["Stato"] = ("Nessun risultato finale" if not any(level_counts) else
                "Baseline top-down non disponibile" if not paired else
                "Parziale" if min(level_counts) < expected or len(set(level_counts)) > 1
                or any(r.get("status") != "completed" for r in planned) else "Disponibile")
            rows.append(row)
    metric_columns = [column for level in ("HLG", "LLG")
                      for column in ([f"{level} {m}" for m in ("Precision", "Recall", "F1")]
                                     + [f"{level} Delta {m}" for m in ("Precision", "Recall", "F1")])]
    return pd.DataFrame(rows, columns=["Esperimento", "Dataset", "Esempi evaluator",
        "Temperatura evaluator", *metric_columns, "Ripetizioni valutate", "Stato", "Suite", "Baseline"])


def convergence_goal_counts_table(collection):
    """Report final counts and convergence per run, without averaging repetitions."""
    def run_key(record):
        return tuple(record.get(field) for field in
                     ("method", "dataset", "baseline_key", "repetition"))

    baselines = {(e["dataset"], e["baseline_key"]): e
                 for e in collection["entries"] if e["method"] == "top_down"}
    entries = {run_key(e): e for e in collection["entries"] if e["method"] != "top_down"}
    runs = {run_key(r): r for r in collection["availability"] if r["method"] != "top_down"}
    for key, entry in entries.items():
        runs.setdefault(key, {**entry, "status": "completed"})
    rows = []
    for key, record in sorted(runs.items(), key=lambda item: tuple(str(v) for v in item[0])):
        method, dataset, baseline_key, repetition = key
        entry = entries.get(key)
        baseline = baselines.get((dataset, baseline_key))
        cycle = entry["payload"].get("globalGoalCycle", {}) if entry else {}
        converged = cycle.get("converged")
        row = {"Esperimento": "Bottom-up originale" if method == "bottom_up" else method.split("/", 1)[-1],
               "Dataset": dataset, "Ripetizione": repetition,
               "Stato run": record.get("status", "unavailable"),
               "Convergenza": "Si" if converged is True else "No" if converged is False else "Non disponibile",
               "Iterazioni completate": cycle.get("completed_iterations"),
               "Motivo arresto": cycle.get("stop_reason"),
               "Suite": record.get("suite_id", ""), "Baseline": baseline_key}
        for level, field in (("HLG", "goals"), ("LLG", "low_level_goals")):
            initial = len(goal_texts(baseline[level.lower()], field)) if baseline else None
            final = len(goal_texts(entry[level.lower()], field)) if entry else None
            row[f"{level} top-down"] = initial
            row[f"{level} bottom-up"] = final
            row[f"Delta numero {level}"] = final - initial if initial is not None and final is not None else None
        rows.append(row)
    return pd.DataFrame(rows, columns=["Esperimento", "Dataset", "Ripetizione", "Stato run",
        "Convergenza", "Iterazioni completate", "Motivo arresto", "HLG top-down", "HLG bottom-up",
        "Delta numero HLG", "LLG top-down", "LLG bottom-up", "Delta numero LLG", "Suite", "Baseline"])


def evaluator_decision_counts_table(collection):
    """Count accepted global-response actions once per model/iteration/branch."""
    actions = ["KEEP_ORIGINAL_HIGH_LEVEL_GOAL", "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL",
               "REMOVE_ORIGINAL_HIGH_LEVEL_GOAL", "DUPLICATE_REMOVAL"]
    labels = dict(zip(actions, ["KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL"]))
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in collection["configurations"]}
    rows = []
    for entry in collection["entries"]:
        if entry["method"] == "top_down":
            continue
        cycle = entry["payload"].get("globalGoalCycle", {})
        settings = {**configs.get(entry["method"], {}), **cycle.get("configuration", {})}
        traces = [t for t in cycle.get("iterations", []) if t.get("checkpoint_complete", True)]
        models = list(dict.fromkeys(list(settings.get("evaluator_models", [])) +
            [v["model"] for t in traces for v in t.get("evaluator_responses", [])]))
        for model in models:
            counts = {action: 0 for action in actions}
            valid, invalid, rejected_attempts, missing, other = 0, 0, 0, 0, 0
            for trace in traces:
                records = [v for v in trace.get("evaluator_responses", []) if v["model"] == model]
                if len(records) > 1:
                    raise ValueError(f"Duplicate global vote for {model}, iteration {trace['iteration']}")
                if not records:
                    missing += 1
                    continue
                record = records[0]
                rejected_attempts += sum(a.get("generation", {}).get("accepted") is False
                                         for a in record.get("attempts", []))
                if record.get("status") != "VALID":
                    invalid += 1
                    continue
                valid += 1
                decisions = record.get("decisions", [])
                ids = [item["branch_id"] for item in decisions]
                if len(ids) != len(set(ids)):
                    raise ValueError("Repeated branch IDs in a saved accepted global response")
                for item in decisions:
                    action = item["decision"]
                    if action in counts:
                        counts[action] += 1
                    else:
                        other += 1
            row = {"Esperimento": "Bottom-up originale" if entry["method"] == "bottom_up"
                   else entry["config_id"], "Dataset": entry["dataset"], "LLM": model,
                   "Ripetizione": entry["repetition"],
                   "Esempi evaluator": settings.get("evaluator_few_shot_examples"),
                   "Temperatura evaluator": settings.get("evaluator_temperature"),
                   "Iterazioni esaminate": len(traces), "Risposte globali valide": valid,
                   "Risposte globali non valide": invalid, "Iterazioni senza risposta registrata": missing,
                   "Tentativi scartati": rejected_attempts,
                   "Totale decisioni valide": sum(counts.values()) + other,
                   "Altre decisioni": other, "Policy": settings.get("pipeline_policy_version"),
                   "Stato": "Nessuna traccia globale" if not valid and not invalid else
                       "Traccia parziale" if missing else "Disponibile",
                   "Suite": entry["suite_id"], "Baseline": entry["baseline_key"]}
            row.update({labels[action]: counts[action] if valid or invalid else None for action in actions})
            if not valid and not invalid:
                row["Totale decisioni valide"] = None
                row["Altre decisioni"] = None
                row["Tentativi scartati"] = None
            rows.append(row)
    columns = ["Esperimento", "Dataset", "LLM", "Ripetizione", "Esempi evaluator",
        "Temperatura evaluator", "KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL",
        "Totale decisioni valide", "Iterazioni esaminate", "Risposte globali valide",
        "Risposte globali non valide", "Tentativi scartati", "Iterazioni senza risposta registrata",
        "Altre decisioni", "Stato", "Policy", "Suite", "Baseline"]
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["Esperimento", "Dataset", "Ripetizione", "LLM"], ignore_index=True)


def final_decision_counts_table(collection):
    """Count approved actions separately from the success of their application."""
    labels = {"KEEP_ORIGINAL_HIGH_LEVEL_GOAL": "KEEP",
              "REWRITE_ORIGINAL_HIGH_LEVEL_GOAL": "REWRITE",
              "REMOVE_ORIGINAL_HIGH_LEVEL_GOAL": "REMOVE", "DUPLICATE_REMOVAL": "DUPLICATE_REMOVAL"}
    configs = {f"{c['suite_id']}/{c['config_id']}": c for c in collection["configurations"]}
    rows = []
    for entry in collection["entries"]:
        if entry["method"] == "top_down":
            continue
        cycle = entry["payload"].get("globalGoalCycle", {})
        settings = {**configs.get(entry["method"], {}), **cycle.get("configuration", {})}
        traces = [t for t in cycle.get("iterations", []) if t.get("checkpoint_complete", True)]
        for scope, selected in (("Intera run", traces), ("Ultima iterazione", traces[-1:])):
            counts = dict.fromkeys(["KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL", "INCONCLUSIVE",
                "Totale decisioni", "Riparazioni completate", "Riparazioni rifiutate", "Rami rimossi"], 0)
            for trace in selected:
                outcomes = trace.get("structural_action_outcomes", {})
                for bid, decision in trace.get("evaluations", {}).items():
                    action = decision.get("decision")
                    if action is not None and action not in labels:
                        raise ValueError(f"Unsupported decision in schema-3 result: {action}")
                    counts[labels[action] if action else "INCONCLUSIVE"] += 1
                    counts["Totale decisioni"] += 1
                    outcome = outcomes.get(bid)
                    for expected, column in (("HLG_LLG_HANDOFF_COMPLETED", "Riparazioni completate"),
                        ("REJECTED_STATE_PRESERVED", "Riparazioni rifiutate"),
                        ("REMOVED_WITH_CHILDREN", "Rami rimossi")):
                        counts[column] += outcome == expected
            rows.append({"Esperimento": "Bottom-up originale" if entry["method"] == "bottom_up"
                         else entry["config_id"], "Dataset": entry["dataset"],
                "Ripetizione": entry["repetition"], "Ambito": scope,
                "Esempi evaluator": settings.get("evaluator_few_shot_examples"),
                "Temperatura evaluator": settings.get("evaluator_temperature"),
                "Iterazioni considerate": len(selected), **{k: v if selected else None for k, v in counts.items()},
                "Stato": "Disponibile" if selected else "Nessuna traccia",
                "Suite": entry["suite_id"], "Baseline": entry["baseline_key"]})
    columns = ["Esperimento", "Dataset", "Ripetizione", "Ambito", "Esempi evaluator",
        "Temperatura evaluator", "Iterazioni considerate", "KEEP", "REWRITE", "REMOVE",
        "DUPLICATE_REMOVAL", "INCONCLUSIVE", "Totale decisioni", "Riparazioni completate",
        "Riparazioni rifiutate", "Rami rimossi", "Stato", "Suite", "Baseline"]
    return pd.DataFrame(rows, columns=columns).sort_values(
        ["Esperimento", "Ambito", "Dataset", "Ripetizione"], ignore_index=True)


def export_decisions_by_configuration(report_dir, tables):
    """Keep combined exports plus individual configuration files for decision tables."""
    root = Path(report_dir) / "decisions_by_configuration"
    for name, table in tables.items():
        for (suite, config), frame in table.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
            identity = f"{suite}/{config}"
            slug = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(config))[:80]
            directory = root / f"{slug}_{digest(identity.encode())[:8]}"
            directory.mkdir(parents=True, exist_ok=True)
            frame.to_csv(directory / f"{name}.csv", index=False, encoding="utf-8-sig")
            frame.to_html(directory / f"{name}.html", index=False, na_rep="-")


def completed_corpus_comparison_table(metrics, configurations, availability, expected_datasets=12):
    """Compare only fully completed configurations on exactly the frozen corpus."""
    levels = {"HLG", "LLG"}
    base = metrics[metrics.method.eq("top_down")]
    baseline_levels = base.groupby(["dataset", "baseline_key"])["level"].agg(set)
    target = {key for key, found in baseline_levels.items() if found == levels}
    if len(target) != expected_datasets or len({d for d, _ in target}) != expected_datasets:
        raise ValueError(f"Servono {expected_datasets} dataset top-down distinti con entrambi i livelli e una sola baseline; trovati {len(target)}")
    eligible, audit = [], []
    for config in configurations:
        method = f"{config['suite_id']}/{config['config_id']}"
        repetitions = int(config.get("planned_repetitions", 1))
        expected = {(d, b, r) for d, b in target for r in range(1, repetitions + 1)}
        planned = [r for r in availability if r.get("method") == method]
        states = {}
        for record in planned:
            key = (record["dataset"], record["baseline_key"], record["repetition"])
            if key in states:
                raise ValueError(f"Run duplicata nei registri: {method}, {key}")
            states[key] = record.get("status")
        completed = {key for key, status in states.items() if status == "completed"}
        frame = metrics[metrics.method.eq(method)]
        metric_keys = list(frame[["dataset", "baseline_key", "repetition", "level"]].itertuples(index=False, name=None))
        expected_metrics = {(*key, level) for key in expected for level in levels}
        evaluated = {(d, b, r) for d, b, r in expected
                     if all((d, b, r, level) in set(metric_keys) for level in levels)}
        finite = frame[METRICS].apply(lambda col: col.map(
            lambda value: pd.notna(value) and float("-inf") < float(value) < float("inf"))).all().all()
        accepted = (repetitions >= 1 and set(states) == expected and completed == expected
                    and set(metric_keys) == expected_metrics and len(metric_keys) == len(expected_metrics) and finite)
        if accepted:
            eligible.append(config)
        audit.append({"Esperimento": config["config_id"], "Suite": config["suite_id"],
            "Dataset finali valutati": len({(d, b) for d, b in target
                if all((d, b, r) in completed & evaluated for r in range(1, repetitions + 1))}),
            "Run concluse previste": len(expected & completed), "Run richieste": len(expected),
            "Incluso sui 12 dataset": accepted,
            "Motivo": "Completa: output finali HLG e LLG per ogni ripetizione" if accepted else
                "Esclusa: run non tutte concluse, metriche mancanti/non valide o baseline non coincidenti"})
    methods = {f"{c['suite_id']}/{c['config_id']}" for c in eligible} | {"top_down"}
    selected = metrics[metrics.method.isin(methods)]
    table, cohort = final_comparison_table(selected, eligible,
        [r for r in availability if r.get("method") in methods])
    if cohort != target or not table["Dataset valutati"].eq(expected_datasets).all():
        raise ValueError("Il confronto completo non deve restringere la coorte")
    return table, cohort, pd.DataFrame(audit)


In [5]:
collection = collect_outputs(PROJECT_ROOT, SUITE_DIRS, historical_suite_id=HISTORICAL_SUITE_ID)
# Considera solo le baseline della campagna selezionata, anche per gli originali.
suite_names = {p.name for p in SUITE_DIRS}
planned_keys = {(r["dataset"], r["baseline_key"]) for r in collection["availability"]
                if r.get("suite_id") in suite_names}
collection["entries"] = [e for e in collection["entries"]
                         if (e["dataset"], e["baseline_key"]) in planned_keys]
collection["availability"] = [r for r in collection["availability"]
    if (r.get("dataset"), r.get("baseline_key")) in planned_keys]
print("Run completate da valutare:", len(collection["entries"]))
print("Dataset previsti:", len(planned_keys))


c:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Run completate da valutare: 72
Dataset previsti: 12


## Metriche sugli output finali

BERT e preprocessing NLTK della valutazione originale; matching uno-a-uno senza soglia. La prima esecuzione richiede che modello e risorse siano disponibili.


In [6]:
from src.evaluation.goal_evaluator import GoalEvaluator

evaluator = GoalEvaluator(model_name="bert-base-uncased", preprocess=True)
METRIC_CONFIGURATION = {
    "model": "bert-base-uncased",
    "resolved_revision": getattr(evaluator.model.config, "_commit_hash", None),
    "goal_text": "description", "preprocessing": "original NLTK",
    "matching": "maximum-weight one-to-one; every assigned cosine contributes",
    "similarity_threshold": None,
    "aggregation": "Arithmetic mean of final-run P, R and F1; repetitions first within dataset, then equal dataset weights",
    "delta_policy": "Experiment mean minus top-down mean on identical paired dataset/baseline cohort; absolute difference",
    "iteration_policy": "Only finalHighLevelGoals and finalLowLevelGoals; no intermediate checkpoint scoring",
}
metrics, matches = evaluate_final_outputs(collection, GROUNDTRUTH_BY_NAME, evaluator.score_goals)
comparison_partial, partial_common_cohort = final_comparison_table(
    metrics, collection["configurations"], collection["availability"])
comparison, common_cohort, completion_12 = completed_corpus_comparison_table(
    metrics, collection["configurations"], collection["availability"], expected_datasets=12)

dataset_comparison = dataset_comparison_table(
    metrics, collection["configurations"], collection["availability"])


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2717.23it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Diagnostica della coorte ridotta (include configurazioni parziali)

Questa vista conserva il confronto sulle configurazioni che hanno almeno un risultato finale. Una configurazione parziale può ridurla a sette o otto dataset: **non è la tabella generale sui dodici dataset**, riportata nella sezione successiva.


In [7]:
# Tabella riepilogativa: valori tra 0 e 1; celle vuote = risultato non disponibile.
metric_columns = [f"{level} {metric}" for level in ("HLG", "LLG") for metric in ("Precision", "Recall", "F1")]
delta_columns = [f"{level} Delta {metric}" for level in ("HLG", "LLG") for metric in ("Precision", "Recall", "F1")]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(comparison_partial.style.format({**{c: "{:.4f}" for c in metric_columns}, **{c: "{:+.4f}" for c in delta_columns}}, na_rep="-").hide(axis="index"))
print("Dataset comuni:", ", ".join(sorted(name for name, _ in partial_common_cohort)) or "nessuno")
print("Le medie usano gli output finali, non la media delle iterazioni intermedie.")
if any(comparison_partial.Stato != "Disponibile"):
    print("Campagna incompleta o coorte ridotta: consultare Stato e numerosita nella tabella.")
print("Delta = esperimento - top-down, sugli stessi dataset: positivo indica incremento, negativo diminuzione.")


Esperimento,Esempi evaluator,Temperatura evaluator,HLG Precision,HLG Recall,HLG F1,HLG Delta Precision,HLG Delta Recall,HLG Delta F1,LLG Precision,LLG Recall,LLG F1,LLG Delta Precision,LLG Delta Recall,LLG Delta F1,Dataset valutati,Dataset disponibili,Run valutate,Stato
Top-down originale,-,-,0.5794,0.5725,0.5297,+0.0000,+0.0000,+0.0000,0.6866,0.4675,0.5083,+0.0000,+0.0000,+0.0000,12,12,12,Disponibile
reference,4.000000,0.200000,0.5596,0.5756,0.5207,-0.0198,+0.0031,-0.0090,0.6710,0.4946,0.5111,-0.0156,+0.0271,+0.0027,12,12,12,Disponibile
E1_0_examples,0.000000,0.200000,0.6017,0.5620,0.5422,+0.0223,-0.0104,+0.0125,0.6863,0.5065,0.5455,-0.0002,+0.0390,+0.0371,12,12,12,Disponibile
E1_1_examples,1.000000,0.200000,0.5958,0.5456,0.5271,+0.0164,-0.0269,-0.0026,0.6726,0.5190,0.5407,-0.0139,+0.0515,+0.0324,12,12,12,Disponibile
E2_temperature_0.0,4.000000,0.000000,0.5616,0.5703,0.5210,-0.0178,-0.0022,-0.0087,0.6645,0.5206,0.5341,-0.0221,+0.0531,+0.0257,12,12,12,Disponibile
E2_temperature_0.4,4.000000,0.400000,0.5847,0.5765,0.5400,+0.0053,+0.0040,+0.0103,0.6559,0.5119,0.5205,-0.0306,+0.0444,+0.0122,12,12,12,Disponibile


Dataset comuni: Assegno Unico Universale - SIA Project 24 25, Calculator, Ethical Purchasing Group - SIA Project 22 23, Event Organization Portal - SIA Project 21 22, Fitness Tracker, Genome Nexus, Gestao Hospital, La Reine Marlene - SIA Project 23 24, London Ambulance Service, Planning Poker, Recycling, SIA Project 25 26
Le medie usano gli output finali, non la media delle iterazioni intermedie.
Delta = esperimento - top-down, sugli stessi dataset: positivo indica incremento, negativo diminuzione.


## Risultati generali: solo configurazioni concluse su tutti i 12 dataset

Questa è la tabella generale da usare per il confronto sul corpus completo. Una configurazione entra soltanto quando **tutte le run previste sui dodici dataset e tutte le ripetizioni sono concluse**, con metriche finali HLG e LLG e baseline coincidenti con il top-down. Le run concluse al limite iterativo restano incluse anche se non convergenti.

Le configurazioni incomplete sono escluse interamente, senza restringere la coorte e senza sostituire risultati mancanti con zeri. Quando anche una configurazione di temperatura completa il corpus, viene inclusa automaticamente dopo aver aggiornato raccolta degli output e metriche. Le ripetizioni sono mediate prima entro dataset; successivamente ogni dataset pesa 1/12. Non occorrono nuove inferenze LLM per costruire questa tabella.

Se il kernel era già aperto prima dell'aggiunta della sezione, rieseguire la cella delle funzioni di valutazione (quella che definisce `METRICS`) prima della cella seguente. `metrics` può essere riutilizzato: non serve ricalcolare BERT se contiene già tutti gli output conclusi desiderati.


In [8]:
comparison_partial, partial_common_cohort = final_comparison_table(
    metrics, collection["configurations"], collection["availability"])
comparison, common_cohort, completion_12 = completed_corpus_comparison_table(
    metrics, collection["configurations"], collection["availability"], expected_datasets=12)
comparison_12 = comparison.copy()
assert len(common_cohort) == 12
assert comparison_12["Dataset valutati"].eq(12).all()
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(comparison_12.style.format({
        **{c: "{:.4f}" for c in metric_columns},
        **{c: "{:+.4f}" for c in delta_columns}}, na_rep="-").hide(axis="index"))
    display(completion_12.drop(columns=["Suite"]))
print("Dataset valutati per ciascun metodo: 12")
print("Dataset:", ", ".join(sorted(name for name, _ in common_cohort)))
print("Configurazioni escluse:", ", ".join(completion_12.loc[
    ~completion_12["Incluso sui 12 dataset"], "Esperimento"]) or "nessuna")


Esperimento,Esempi evaluator,Temperatura evaluator,HLG Precision,HLG Recall,HLG F1,HLG Delta Precision,HLG Delta Recall,HLG Delta F1,LLG Precision,LLG Recall,LLG F1,LLG Delta Precision,LLG Delta Recall,LLG Delta F1,Dataset valutati,Dataset disponibili,Run valutate,Stato
Top-down originale,-,-,0.5794,0.5725,0.5297,+0.0000,+0.0000,+0.0000,0.6866,0.4675,0.5083,+0.0000,+0.0000,+0.0000,12,12,12,Disponibile
reference,4.000000,0.200000,0.5596,0.5756,0.5207,-0.0198,+0.0031,-0.0090,0.6710,0.4946,0.5111,-0.0156,+0.0271,+0.0027,12,12,12,Disponibile
E1_0_examples,0.000000,0.200000,0.6017,0.5620,0.5422,+0.0223,-0.0104,+0.0125,0.6863,0.5065,0.5455,-0.0002,+0.0390,+0.0371,12,12,12,Disponibile
E1_1_examples,1.000000,0.200000,0.5958,0.5456,0.5271,+0.0164,-0.0269,-0.0026,0.6726,0.5190,0.5407,-0.0139,+0.0515,+0.0324,12,12,12,Disponibile
E2_temperature_0.0,4.000000,0.000000,0.5616,0.5703,0.5210,-0.0178,-0.0022,-0.0087,0.6645,0.5206,0.5341,-0.0221,+0.0531,+0.0257,12,12,12,Disponibile
E2_temperature_0.4,4.000000,0.400000,0.5847,0.5765,0.5400,+0.0053,+0.0040,+0.0103,0.6559,0.5119,0.5205,-0.0306,+0.0444,+0.0122,12,12,12,Disponibile


,Esperimento,Dataset finali valutati,Run concluse previste,Run richieste,Incluso sui 12 dataset,Motivo
0,reference,12,12,12,True,Completa: output finali HLG e LLG per ogni rip...
1,E1_0_examples,12,12,12,True,Completa: output finali HLG e LLG per ogni rip...
2,E1_1_examples,12,12,12,True,Completa: output finali HLG e LLG per ogni rip...
3,E2_temperature_0.0,12,12,12,True,Completa: output finali HLG e LLG per ogni rip...
4,E2_temperature_0.4,12,12,12,True,Completa: output finali HLG e LLG per ogni rip...


Dataset valutati per ciascun metodo: 12
Dataset: Assegno Unico Universale - SIA Project 24 25, Calculator, Ethical Purchasing Group - SIA Project 22 23, Event Organization Portal - SIA Project 21 22, Fitness Tracker, Genome Nexus, Gestao Hospital, La Reine Marlene - SIA Project 23 24, London Ambulance Service, Planning Poker, Recycling, SIA Project 25 26
Configurazioni escluse: nessuna


## Confronto per esperimento e dataset

Una sola tabella contiene tutte le configurazioni e i dataset, con le metriche HLG e LLG e i delta appaiati: Delta = esperimento - top-down dello stesso dataset e della stessa baseline. Le metriche usano il matching BERT uno-a-uno gia calcolato, senza nuove inferenze. Le eventuali ripetizioni vengono mediate separatamente per Precision, Recall e F1.

Questa tabella mostra tutti i dataset pianificati, senza limitarli alla coorte comune della tabella riepilogativa. Risultati mancanti rimangono vuoti; Stato e Ripetizioni valutate indicano la copertura. I delta mancanti non sono sostituiti con zero.


In [9]:
# Suite e hash della baseline sono conservati negli export per la provenienza.
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(dataset_comparison.drop(columns=["Suite", "Baseline"]).style.format(
        {**{c: "{:.4f}" for c in metric_columns},
         **{c: "{:+.4f}" for c in delta_columns}}, na_rep="-").hide(axis="index"))
print("Delta per dataset = esperimento - top-down; positivo indica incremento, negativo diminuzione.")


Esperimento,Dataset,Esempi evaluator,Temperatura evaluator,HLG Precision,HLG Recall,HLG F1,HLG Delta Precision,HLG Delta Recall,HLG Delta F1,LLG Precision,LLG Recall,LLG F1,LLG Delta Precision,LLG Delta Recall,LLG Delta F1,Ripetizioni valutate,Stato
Top-down originale,Assegno Unico Universale - SIA Project 24 25,-,-,0.3104,0.7760,0.4435,+0.0000,+0.0000,+0.0000,0.7838,0.6719,0.7235,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Calculator,-,-,0.8048,0.4024,0.5366,+0.0000,+0.0000,+0.0000,0.7299,0.5214,0.6083,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Ethical Purchasing Group - SIA Project 22 23,-,-,0.3009,0.6771,0.4167,+0.0000,+0.0000,+0.0000,0.4162,0.6540,0.5087,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Event Organization Portal - SIA Project 21 22,-,-,0.6454,0.4840,0.5532,+0.0000,+0.0000,+0.0000,0.7325,0.2616,0.3855,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Fitness Tracker,-,-,0.5798,0.6765,0.6244,+0.0000,+0.0000,+0.0000,0.5548,0.7255,0.6288,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Genome Nexus,-,-,0.6785,0.4524,0.5428,+0.0000,+0.0000,+0.0000,0.8608,0.2296,0.3625,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Gestao Hospital,-,-,0.6135,0.7669,0.6817,+0.0000,+0.0000,+0.0000,0.7391,0.5338,0.6199,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,La Reine Marlene - SIA Project 23 24,-,-,0.7148,0.2145,0.3299,+0.0000,+0.0000,+0.0000,0.7649,0.2638,0.3923,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,London Ambulance Service,-,-,0.3821,0.7643,0.5095,+0.0000,+0.0000,+0.0000,0.3631,0.7262,0.4841,+0.0000,+0.0000,+0.0000,1,Disponibile
Top-down originale,Planning Poker,-,-,0.6863,0.4118,0.5147,+0.0000,+0.0000,+0.0000,0.7294,0.1989,0.3126,+0.0000,+0.0000,+0.0000,1,Disponibile


Delta per dataset = esperimento - top-down; positivo indica incremento, negativo diminuzione.


## Convergenza e numero di goal

Una riga per dataset, esperimento e ripetizione: convergenza, motivo di arresto, iterazioni completate e numero di HLG e LLG nella baseline top-down e nella gerarchia finale bottom-up. I delta dei conteggi sono bottom-up meno top-down.

Le ripetizioni restano separate. Le run completate senza convergenza mantengono i conteggi finali; per run non completate i risultati finali e la convergenza restano non disponibili, senza utilizzare checkpoint intermedi. Una collezione finale vuota vale zero; un risultato mancante rimane vuoto.


In [10]:
convergence_goal_counts = convergence_goal_counts_table(collection)
count_columns = ["Ripetizione", "Iterazioni completate", "HLG top-down", "HLG bottom-up",
                 "LLG top-down", "LLG bottom-up"]
count_deltas = ["Delta numero HLG", "Delta numero LLG"]
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(convergence_goal_counts.drop(columns=["Suite", "Baseline"]).style.format(
        {**{c: "{:.0f}" for c in count_columns},
         **{c: "{:+.0f}" for c in count_deltas}}, na_rep="-").hide(axis="index"))


Esperimento,Dataset,Ripetizione,Stato run,Convergenza,Iterazioni completate,Motivo arresto,HLG top-down,HLG bottom-up,Delta numero HLG,LLG top-down,LLG bottom-up,Delta numero LLG
E1_0_examples,Assegno Unico Universale - SIA Project 24 25,1,completed,No,25,MAX_ITERATIONS_REACHED,10,8,-2,24,23,-1
E1_0_examples,Calculator,1,completed,Si,1,ALL_BRANCHES_CONFIRMED,1,1,+0,5,5,+0
E1_0_examples,Ethical Purchasing Group - SIA Project 22 23,1,completed,Si,1,ALL_BRANCHES_CONFIRMED,9,9,+0,22,22,+0
E1_0_examples,Event Organization Portal - SIA Project 21 22,1,completed,Si,3,ALL_BRANCHES_CONFIRMED,3,3,+0,10,20,+10
E1_0_examples,Fitness Tracker,1,completed,Si,6,ALL_BRANCHES_CONFIRMED,7,6,-1,17,15,-2
E1_0_examples,Genome Nexus,1,completed,Si,1,ALL_BRANCHES_CONFIRMED,2,2,+0,8,8,+0
E1_0_examples,Gestao Hospital,1,completed,Si,1,ALL_BRANCHES_CONFIRMED,5,5,+0,13,13,+0
E1_0_examples,La Reine Marlene - SIA Project 23 24,1,completed,Si,7,ALL_BRANCHES_CONFIRMED,3,3,+0,10,14,+4
E1_0_examples,London Ambulance Service,1,completed,Si,13,ALL_BRANCHES_CONFIRMED,4,4,+0,6,6,+0
E1_0_examples,Planning Poker,1,completed,No,25,MAX_ITERATIONS_REACHED,3,4,+1,9,10,+1


## Decisioni dei singoli LLM per progetto e configurazione

Una riga per esperimento, dataset, ripetizione e LLM, con il numero di esempi e la temperatura effettivamente registrati. Le colonne KEEP, REWRITE, REMOVE e DUPLICATE_REMOVAL contano le decisioni dei singoli evaluator nelle risposte globali valide, sommando tutte le iterazioni completate della run. Uno stesso goal puo essere valutato piu volte: i conteggi rappresentano decisioni per ramo e iterazione, non goal distinti.

Sono proposte dei singoli modelli, non interventi applicati dall'orchestratore. Le risposte scartate non contribuiscono alle quattro azioni; risposte globali non valide e tentativi scartati sono riportati separatamente. Un retry scartato viene contato una volta per modello e iterazione, senza moltiplicarlo per il numero di branch. Le run non completate non entrano in questa analisi, coerentemente con la raccolta degli output finali. Eventuali tracce mancanti sono segnalate.

La visualizzazione e suddivisa in una tabella per configurazione; i file completi mantengono tutte le configurazioni e vengono affiancati dagli export separati.


In [11]:
evaluator_decision_counts = evaluator_decision_counts_table(collection)
llm_count_columns = ["Ripetizione", "Esempi evaluator", "KEEP", "REWRITE", "REMOVE", "DUPLICATE_REMOVAL",
    "Totale decisioni valide", "Iterazioni esaminate", "Risposte globali valide",
    "Risposte globali non valide", "Tentativi scartati", "Iterazioni senza risposta registrata", "Altre decisioni"]
for (suite, config), frame in evaluator_decision_counts.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
    settings = frame.iloc[0]
    print(f"Configurazione: {config} | esempi: {settings['Esempi evaluator']} | temperatura: {settings['Temperatura evaluator']}")
    with pd.option_context("display.max_columns", None, "display.max_rows", None):
        display(frame.drop(columns=["Suite", "Baseline", "Esperimento"]).style.format(
            {**{c: "{:.0f}" for c in llm_count_columns}, "Temperatura evaluator": "{:.1f}"},
            na_rep="-").hide(axis="index"))


Configurazione: E1_0_examples | esempi: 0 | temperatura: 0.2


Dataset,LLM,Ripetizione,Esempi evaluator,Temperatura evaluator,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,Totale decisioni valide,Iterazioni esaminate,Risposte globali valide,Risposte globali non valide,Tentativi scartati,Iterazioni senza risposta registrata,Altre decisioni,Stato,Policy
Assegno Unico Universale - SIA Project 24 25,google/gemma-4-12b-qat,1,0,0.2,162,18,0,2,182,25,25,0,1,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,ibm/granite-3.2-8b,1,0,0.2,177,5,0,0,182,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,mistralai/ministral-3-14b-reasoning,1,0,0.2,126,54,0,2,182,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,google/gemma-4-12b-qat,1,0,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,ibm/granite-3.2-8b,1,0,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,mistralai/ministral-3-14b-reasoning,1,0,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,google/gemma-4-12b-qat,1,0,0.2,8,1,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,ibm/granite-3.2-8b,1,0,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,mistralai/ministral-3-14b-reasoning,1,0,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Event Organization Portal - SIA Project 21 22,google/gemma-4-12b-qat,1,0,0.2,6,3,0,0,9,3,3,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs


Configurazione: E1_1_examples | esempi: 1 | temperatura: 0.2


Dataset,LLM,Ripetizione,Esempi evaluator,Temperatura evaluator,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,Totale decisioni valide,Iterazioni esaminate,Risposte globali valide,Risposte globali non valide,Tentativi scartati,Iterazioni senza risposta registrata,Altre decisioni,Stato,Policy
Assegno Unico Universale - SIA Project 24 25,google/gemma-4-12b-qat,1,1,0.2,91,64,0,3,158,21,21,0,4,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,ibm/granite-3.2-8b,1,1,0.2,105,48,1,4,158,21,21,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,mistralai/ministral-3-14b-reasoning,1,1,0.2,84,67,1,6,158,21,21,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,google/gemma-4-12b-qat,1,1,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,ibm/granite-3.2-8b,1,1,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,mistralai/ministral-3-14b-reasoning,1,1,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,google/gemma-4-12b-qat,1,1,0.2,8,1,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,ibm/granite-3.2-8b,1,1,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,mistralai/ministral-3-14b-reasoning,1,1,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Event Organization Portal - SIA Project 21 22,google/gemma-4-12b-qat,1,1,0.2,15,15,0,0,30,9,8,1,3,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs


Configurazione: E2_temperature_0.0 | esempi: 4 | temperatura: 0.0


Dataset,LLM,Ripetizione,Esempi evaluator,Temperatura evaluator,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,Totale decisioni valide,Iterazioni esaminate,Risposte globali valide,Risposte globali non valide,Tentativi scartati,Iterazioni senza risposta registrata,Altre decisioni,Stato,Policy
Assegno Unico Universale - SIA Project 24 25,google/gemma-4-12b-qat,1,4,0.0,211,86,0,1,298,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,ibm/granite-3.2-8b,1,4,0.0,224,72,0,2,298,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,mistralai/ministral-3-14b-reasoning,1,4,0.0,164,89,42,3,298,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,google/gemma-4-12b-qat,1,4,0.0,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,ibm/granite-3.2-8b,1,4,0.0,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,mistralai/ministral-3-14b-reasoning,1,4,0.0,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,google/gemma-4-12b-qat,1,4,0.0,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,ibm/granite-3.2-8b,1,4,0.0,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,mistralai/ministral-3-14b-reasoning,1,4,0.0,8,1,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Event Organization Portal - SIA Project 21 22,google/gemma-4-12b-qat,1,4,0.0,49,26,0,0,75,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs


Configurazione: E2_temperature_0.4 | esempi: 4 | temperatura: 0.4


Dataset,LLM,Ripetizione,Esempi evaluator,Temperatura evaluator,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,Totale decisioni valide,Iterazioni esaminate,Risposte globali valide,Risposte globali non valide,Tentativi scartati,Iterazioni senza risposta registrata,Altre decisioni,Stato,Policy
Assegno Unico Universale - SIA Project 24 25,google/gemma-4-12b-qat,1,4,0.4,36,17,0,1,54,6,6,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,ibm/granite-3.2-8b,1,4,0.4,48,5,0,1,54,6,6,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,mistralai/ministral-3-14b-reasoning,1,4,0.4,31,20,1,2,54,6,6,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,google/gemma-4-12b-qat,1,4,0.4,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,ibm/granite-3.2-8b,1,4,0.4,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,mistralai/ministral-3-14b-reasoning,1,4,0.4,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,google/gemma-4-12b-qat,1,4,0.4,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,ibm/granite-3.2-8b,1,4,0.4,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,mistralai/ministral-3-14b-reasoning,1,4,0.4,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Event Organization Portal - SIA Project 21 22,google/gemma-4-12b-qat,1,4,0.4,4,2,0,0,6,2,2,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs


Configurazione: reference | esempi: 4 | temperatura: 0.2


Dataset,LLM,Ripetizione,Esempi evaluator,Temperatura evaluator,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,Totale decisioni valide,Iterazioni esaminate,Risposte globali valide,Risposte globali non valide,Tentativi scartati,Iterazioni senza risposta registrata,Altre decisioni,Stato,Policy
Assegno Unico Universale - SIA Project 24 25,google/gemma-4-12b-qat,1,4,0.2,144,86,0,1,231,25,22,3,18,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,ibm/granite-3.2-8b,1,4,0.2,184,70,1,8,263,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Assegno Unico Universale - SIA Project 24 25,mistralai/ministral-3-14b-reasoning,1,4,0.2,152,102,0,9,263,25,25,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,google/gemma-4-12b-qat,1,4,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,ibm/granite-3.2-8b,1,4,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Calculator,mistralai/ministral-3-14b-reasoning,1,4,0.2,1,0,0,0,1,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,google/gemma-4-12b-qat,1,4,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,ibm/granite-3.2-8b,1,4,0.2,9,0,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Ethical Purchasing Group - SIA Project 22 23,mistralai/ministral-3-14b-reasoning,1,4,0.2,8,1,0,0,9,1,1,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs
Event Organization Portal - SIA Project 21 22,google/gemma-4-12b-qat,1,4,0.2,4,2,0,0,6,2,2,0,0,0,0,Disponibile,cosine_global_documentary_review_v111_schematic_inputs


## Decisioni aggregate del ciclo per configurazione

Le tabelle distinguono le quattro azioni approvate e i rami inconcludenti, separatamente per intera run e ultima iterazione completata. L'assenza di maggioranza e rappresentata da `decision=null`, non da una quinta azione LLM. La duplicazione mantiene il valore esatto `DUPLICATE_REMOVAL`.

Gli esiti applicativi sono separati: una riscrittura approvata ma rifiutata durante la riparazione resta una proposta di riscrittura, con esito di fallimento. I conteggi si riferiscono a ramo e iterazione, non a goal unici o chiamate ai modelli. Sono supportate soltanto nuove run con schema 3; le colonne delle policy storiche sono eliminate.


In [12]:
final_decision_counts = final_decision_counts_table(collection)
final_count_columns = ["Ripetizione", "Esempi evaluator", "Iterazioni considerate", "KEEP", "REWRITE", "REMOVE",
    "DUPLICATE_REMOVAL", "INCONCLUSIVE", "Totale decisioni", "Riparazioni completate", "Riparazioni rifiutate",
    "Rami rimossi"]
for (suite, config), frame in final_decision_counts.groupby(["Suite", "Esperimento"], dropna=False, sort=False):
    settings = frame.iloc[0]
    print(f"Configurazione: {config} | esempi: {settings['Esempi evaluator']} | temperatura: {settings['Temperatura evaluator']}")
    for scope, scoped in frame.groupby("Ambito", sort=False):
        print(scope)
        with pd.option_context("display.max_columns", None, "display.max_rows", None):
            display(scoped.drop(columns=["Suite", "Baseline", "Esperimento", "Ambito"]).style.format(
                {**{c: "{:.0f}" for c in final_count_columns}, "Temperatura evaluator": "{:.1f}"},
                na_rep="-").hide(axis="index"))


Configurazione: E1_0_examples | esempi: 0 | temperatura: 0.2
Intera run


Dataset,Ripetizione,Esempi evaluator,Temperatura evaluator,Iterazioni considerate,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,INCONCLUSIVE,Totale decisioni,Riparazioni completate,Riparazioni rifiutate,Rami rimossi,Stato
Assegno Unico Universale - SIA Project 24 25,1,0,0.2,25,163,17,0,2,20,202,17,0,2,Disponibile
Calculator,1,0,0.2,1,1,0,0,0,0,1,0,0,0,Disponibile
Ethical Purchasing Group - SIA Project 22 23,1,0,0.2,1,9,0,0,0,0,9,0,0,0,Disponibile
Event Organization Portal - SIA Project 21 22,1,0,0.2,3,6,3,0,0,0,9,3,0,0,Disponibile
Fitness Tracker,1,0,0.2,6,30,7,0,1,3,41,7,0,1,Disponibile
Genome Nexus,1,0,0.2,1,2,0,0,0,0,2,0,0,0,Disponibile
Gestao Hospital,1,0,0.2,1,5,0,0,0,0,5,0,0,0,Disponibile
La Reine Marlene - SIA Project 23 24,1,0,0.2,7,10,11,0,0,0,21,11,0,0,Disponibile
London Ambulance Service,1,0,0.2,13,33,18,0,0,1,52,18,0,0,Disponibile
Planning Poker,1,0,0.2,25,43,41,0,1,0,85,33,8,1,Disponibile


Ultima iterazione


Dataset,Ripetizione,Esempi evaluator,Temperatura evaluator,Iterazioni considerate,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,INCONCLUSIVE,Totale decisioni,Riparazioni completate,Riparazioni rifiutate,Rami rimossi,Stato
Assegno Unico Universale - SIA Project 24 25,1,0,0.2,1,7,0,0,0,1,8,0,0,0,Disponibile
Calculator,1,0,0.2,1,1,0,0,0,0,1,0,0,0,Disponibile
Ethical Purchasing Group - SIA Project 22 23,1,0,0.2,1,9,0,0,0,0,9,0,0,0,Disponibile
Event Organization Portal - SIA Project 21 22,1,0,0.2,1,3,0,0,0,0,3,0,0,0,Disponibile
Fitness Tracker,1,0,0.2,1,6,0,0,0,0,6,0,0,0,Disponibile
Genome Nexus,1,0,0.2,1,2,0,0,0,0,2,0,0,0,Disponibile
Gestao Hospital,1,0,0.2,1,5,0,0,0,0,5,0,0,0,Disponibile
La Reine Marlene - SIA Project 23 24,1,0,0.2,1,3,0,0,0,0,3,0,0,0,Disponibile
London Ambulance Service,1,0,0.2,1,4,0,0,0,0,4,0,0,0,Disponibile
Planning Poker,1,0,0.2,1,2,2,0,0,0,4,2,0,0,Disponibile


Configurazione: E1_1_examples | esempi: 1 | temperatura: 0.2
Intera run


...

Dataset,Ripetizione,Esempi evaluator,Temperatura evaluator,Iterazioni considerate,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,INCONCLUSIVE,Totale decisioni,Riparazioni completate,Riparazioni rifiutate,Rami rimossi,Stato
Assegno Unico Universale - SIA Project 24 25,1,4,0.4,1,8,0,0,0,0,8,0,0,0,Disponibile
Calculator,1,4,0.4,1,1,0,0,0,0,1,0,0,0,Disponibile
Ethical Purchasing Group - SIA Project 22 23,1,4,0.4,1,9,0,0,0,0,9,0,0,0,Disponibile
Event Organization Portal - SIA Project 21 22,1,4,0.4,1,3,0,0,0,0,3,0,0,0,Disponibile
Fitness Tracker,1,4,0.4,1,5,2,0,0,0,7,2,0,0,Disponibile
Genome Nexus,1,4,0.4,1,2,0,0,0,0,2,0,0,0,Disponibile
Gestao Hospital,1,4,0.4,1,5,0,0,0,0,5,0,0,0,Disponibile
La Reine Marlene - SIA Project 23 24,1,4,0.4,1,3,0,0,0,0,3,0,0,0,Disponibile
London Ambulance Service,1,4,0.4,1,4,0,0,0,0,4,0,0,0,Disponibile
Planning Poker,1,4,0.4,1,3,0,0,0,0,3,0,0,0,Disponibile


Configurazione: reference | esempi: 4 | temperatura: 0.2
Intera run


Dataset,Ripetizione,Esempi evaluator,Temperatura evaluator,Iterazioni considerate,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,INCONCLUSIVE,Totale decisioni,Riparazioni completate,Riparazioni rifiutate,Rami rimossi,Stato
Assegno Unico Universale - SIA Project 24 25,1,4,0.2,25,162,92,0,4,10,268,89,3,4,Disponibile
Calculator,1,4,0.2,1,1,0,0,0,0,1,0,0,0,Disponibile
Ethical Purchasing Group - SIA Project 22 23,1,4,0.2,1,9,0,0,0,0,9,0,0,0,Disponibile
Event Organization Portal - SIA Project 21 22,1,4,0.2,2,4,2,0,0,0,6,2,0,0,Disponibile
Fitness Tracker,1,4,0.2,16,63,34,0,1,12,110,34,0,1,Disponibile
Genome Nexus,1,4,0.2,1,2,0,0,0,0,2,0,0,0,Disponibile
Gestao Hospital,1,4,0.2,1,5,0,0,0,0,5,0,0,0,Disponibile
La Reine Marlene - SIA Project 23 24,1,4,0.2,8,15,9,0,0,0,24,9,0,0,Disponibile
London Ambulance Service,1,4,0.2,5,10,10,0,0,0,20,10,0,0,Disponibile
Planning Poker,1,4,0.2,4,7,5,0,0,0,12,5,0,0,Disponibile


Ultima iterazione


Dataset,Ripetizione,Esempi evaluator,Temperatura evaluator,Iterazioni considerate,KEEP,REWRITE,REMOVE,DUPLICATE_REMOVAL,INCONCLUSIVE,Totale decisioni,Riparazioni completate,Riparazioni rifiutate,Rami rimossi,Stato
Assegno Unico Universale - SIA Project 24 25,1,4,0.2,1,7,3,0,0,0,10,3,0,0,Disponibile
Calculator,1,4,0.2,1,1,0,0,0,0,1,0,0,0,Disponibile
Ethical Purchasing Group - SIA Project 22 23,1,4,0.2,1,9,0,0,0,0,9,0,0,0,Disponibile
Event Organization Portal - SIA Project 21 22,1,4,0.2,1,3,0,0,0,0,3,0,0,0,Disponibile
Fitness Tracker,1,4,0.2,1,7,0,0,0,0,7,0,0,0,Disponibile
Genome Nexus,1,4,0.2,1,2,0,0,0,0,2,0,0,0,Disponibile
Gestao Hospital,1,4,0.2,1,5,0,0,0,0,5,0,0,0,Disponibile
La Reine Marlene - SIA Project 23 24,1,4,0.2,1,3,0,0,0,0,3,0,0,0,Disponibile
London Ambulance Service,1,4,0.2,1,4,0,0,0,0,4,0,0,0,Disponibile
Planning Poker,1,4,0.2,1,3,0,0,0,0,3,0,0,0,Disponibile


## Salvataggio

`final_comparison.csv` e `final_comparison.html` contengono il riepilogo esistente. `dataset_comparison.csv` e `dataset_comparison.html` contengono la tabella unica per esperimento e dataset, incluse metriche, delta e provenienza. I CSV di dettaglio conservano anche copertura e metriche per singola run.

`convergence_goal_counts.csv` e `convergence_goal_counts.html` contengono convergenza e conteggi dei goal per dataset, esperimento e ripetizione.

`evaluator_decision_counts.csv` e `evaluator_decision_counts.html` contengono i conteggi delle decisioni per LLM, progetto e configurazione, con diagnostica e provenienza.

`final_decision_counts.csv` e `.html` riportano le decisioni aggregate e gli esiti applicativi. La cartella `decisions_by_configuration/` conserva inoltre entrambe le tabelle decisionali in file separati per configurazione.

La tabella generale `final_comparison.csv` e la copia esplicita `final_comparison_12.csv` includono soltanto configurazioni complete sui dodici dataset. `configuration_completion_12.csv` documenta inclusioni ed esclusioni; `partial_cohort_comparison.csv` conserva separatamente la diagnostica della coorte ridotta.


In [13]:
# Verifica che gli input non siano cambiati durante il calcolo.
changed = [str(p) for p, checksum in collection["input_hashes"].items()
           if not p.exists() or digest(p.read_bytes()) != checksum]
if changed:
    raise RuntimeError(f"Input modificati durante la valutazione; rieseguire il notebook: {changed}")
REPORT_DIR.mkdir(parents=True, exist_ok=False)
comparison.to_csv(REPORT_DIR / "final_comparison.csv", index=False, encoding="utf-8-sig")
comparison.to_html(REPORT_DIR / "final_comparison.html", index=False, float_format=lambda x: f"{x:.4f}", na_rep="-")
# Tabella generale: solo configurazioni complete sul corpus di 12 dataset.
comparison.to_csv(REPORT_DIR / "final_comparison_12.csv", index=False, encoding="utf-8-sig")
comparison.to_html(REPORT_DIR / "final_comparison_12.html", index=False, float_format=lambda x: f"{x:.4f}", na_rep="-")
completion_12.to_csv(REPORT_DIR / "configuration_completion_12.csv", index=False, encoding="utf-8-sig")
comparison_partial.to_csv(REPORT_DIR / "partial_cohort_comparison.csv", index=False, encoding="utf-8-sig")

dataset_comparison.to_csv(REPORT_DIR / "dataset_comparison.csv", index=False, encoding="utf-8-sig")
dataset_comparison.to_html(REPORT_DIR / "dataset_comparison.html", index=False, float_format=lambda x: f"{x:.4f}", na_rep="-")
convergence_goal_counts.to_csv(REPORT_DIR / "convergence_goal_counts.csv", index=False, encoding="utf-8-sig")
convergence_goal_counts.to_html(REPORT_DIR / "convergence_goal_counts.html", index=False, na_rep="-")
evaluator_decision_counts.to_csv(REPORT_DIR / "evaluator_decision_counts.csv", index=False, encoding="utf-8-sig")
evaluator_decision_counts.to_html(REPORT_DIR / "evaluator_decision_counts.html", index=False, na_rep="-")
final_decision_counts.to_csv(REPORT_DIR / "final_decision_counts.csv", index=False, encoding="utf-8-sig")
final_decision_counts.to_html(REPORT_DIR / "final_decision_counts.html", index=False, na_rep="-")
export_decisions_by_configuration(REPORT_DIR, {
    "evaluator_decision_counts": evaluator_decision_counts,
    "final_decision_counts": final_decision_counts})
metrics.to_csv(REPORT_DIR / "final_metrics.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(collection["availability"]).to_csv(REPORT_DIR / "availability.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(collection["configurations"]).to_csv(REPORT_DIR / "configurations.csv", index=False, encoding="utf-8-sig")
manifest = {
    "suite_id": SUITE_DIRS[0].name, "metric": METRIC_CONFIGURATION,
    "complete_corpus_policy": "Exactly 12 paired datasets; all planned repetitions completed with both final metric levels; partial configurations excluded",
    "partial_common_cohort": [{"dataset": d, "baseline_key": b} for d, b in sorted(partial_common_cohort)],
    "common_cohort": [{"dataset": d, "baseline_key": b} for d, b in sorted(common_cohort)],
    "sources": {str(p): h for p, h in collection["input_hashes"].items()},
    "ground_truth_sha256": digest(json.dumps(GROUNDTRUTH_BY_NAME, sort_keys=True, ensure_ascii=False).encode()),
    "status_policy": "Only completed runs, including nonconvergent; missing/failed runs never replaced by zero",
    "dataset_table_policy": "All planned dataset/baseline pairs; final repetitions averaged within each method; deltas against exact matching top-down baseline; missing values preserved",
    "convergence_table_policy": "Per run; final HLG/LLG counts paired with frozen top-down baseline; repetitions separate; unfinished runs have no final counts or convergence",
    "evaluator_decision_table_policy": "Completed runs; accepted global-response decisions summed across completed iterations per LLM/dataset/configuration/repetition; rejected attempts counted once per global request, not per branch; no applied-action inference",
    "final_decision_table_policy": "Recorded orchestrator decisions and applied branch outcomes; whole run and last iteration separately; per-configuration displays and exports",
    "pairing": "Identical datasets and frozen baselines across methods with available final outputs",
}
(REPORT_DIR / "evaluation_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Tabella salvata:", REPORT_DIR / "final_comparison.csv")
print("Tabella per dataset salvata:", REPORT_DIR / "dataset_comparison.csv")
print("Convergenza e conteggi salvati:", REPORT_DIR / "convergence_goal_counts.csv")
print("Decisioni per LLM salvate:", REPORT_DIR / "evaluator_decision_counts.csv")
print("Decisioni finali e tabelle per configurazione salvate:", REPORT_DIR / "decisions_by_configuration")


Tabella salvata: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\configuration_comparison\20261026_230638_419662\final_comparison.csv
Tabella per dataset salvata: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\configuration_comparison\20261026_230638_419662\dataset_comparison.csv
Convergenza e conteggi salvati: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\configuration_comparison\20261026_230638_419662\convergence_goal_counts.csv
Decisioni per LLM salvate: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlignement\output\evaluation\configuration_comparison\20261026_230638_419662\evaluator_decision_counts.csv
Decisioni finali e tabelle per configurazione salvate: C:\Users\agnes\Desktop\TESI FINALE CON EXPERIMENT\LLM_MultiAgentChain_GoalOrientedAPIAlign